# Quality check: calibrating the screening on the 24 Italian maps

`copernicus_ems.quality` screens an EMS flood map against Sentinel-1: inside the map's observation mask it detects new open water (dark now, not dark in a pre-event pass of the same orbit, patches of at least 4 pixels) and measures how far the radar agrees with the map. Three flags send a map to a human look:

* **possibly partial**: the mapped water spans a small part of the area (`mapped_extent` < `PARTIAL_EXTENT`) **and** the radar sees large unmapped dark patches outside that extent, more than `PARTIAL_RATIO` times the mapped water;
* **much unmapped water**: large unmapped dark patches away from the mapped water, more than `UNMAPPED_RATIO` times the mapped water;
* **little confirmed**: the radar confirms less than `LITTLE_CONFIRMED` of the mapped water.

The four thresholds are set here, on the 24 maps of notebooks 01–07. The rule (the design's, the owner's decision): **at most a fifth of the assessable maps "to check"**; within that cap, flag the known suspect, EMSR664 AOI01 Pléiades (notebook 07), as possibly partial. If the two aims conflict the cap wins: the weaker flags are tightened first, and the suspect may stay unflagged.

With one known suspect this is a screening, not a validated classifier. A low agreement is normal for maps drawn on other sensors and dates (notebook 01); a verdict says where to look, never that a map is mistaken.

**0 PU**: it reads the pairs and references of notebook 02.

In [ ]:
import math
import os
from pathlib import Path
from statistics import median
from unittest import mock

from copernicus_ems import quality
from copernicus_ems.dataset import find_pairs

DATA_DIR = Path(os.environ.get("COPERNICUS_EMS_DATA_DIR", "~/.cache/copernicus-ems")).expanduser()
FIGURE = Path(os.environ.get("NOTEBOOK_FIGURE", "../docs/images/notebook-quality-s1.png"))
SUSPECT = ("EMSR664", "AOI01_GRA_PRODUCT")

pairs = list(find_pairs(DATA_DIR))
reports = [quality.check_pair(p) for p in pairs]  # the same as check_pairs(DATA_DIR, write=False)
assert reports, "no pair found: run notebook 02 first"
rows = []
for p, r in zip(pairs, reports, strict=True):
    sensors = sorted({s.get("sensor", "") for s in p.meta.get("made_from") or []})
    rows.append({"pair": p, "report": r, "sensors": ", ".join(sensors)})
assessable = [x for x in rows if x["report"].verdict != quality.NOT_ASSESSABLE]
CAP = math.ceil(len(assessable) / 5)
print(
    f"{len(rows)} maps, {sum(r.same_image for r in reports)} drawn on Sentinel-1, "
    f"{len(assessable)} assessable: at most {CAP} 'to check'"
)
print(
    "thresholds in the module:",
    f"PARTIAL_EXTENT={quality.PARTIAL_EXTENT}, PARTIAL_RATIO={quality.PARTIAL_RATIO},",
    f"UNMAPPED_RATIO={quality.UNMAPPED_RATIO}, LITTLE_CONFIRMED={quality.LITTLE_CONFIRMED}",
)

## 1. Every map, its measurements and its verdict

*confirmed*: share of the mapped water the radar sees as new water. *unmapped clustered*: unmapped detected water in patches of at least 1 km², more than 120 m from the mapped water. *extent*: the mapped water's bounding box as a share of the area's. *outside*: clustered unmapped water outside that box.

In [ ]:
def line(x):
    r, m = x["report"], x["report"].measurements
    flags = ", ".join(f.code for f in r.flags) or (r.reason or "")
    return (
        f"{r.activation:8} {r.product:18} {x['sensors'][:22]:22} {'yes' if r.same_image else 'no':4} "
        f"{r.gap_hours:+6.1f} {m.coverage:5.2f} {m.mapped_km2:7.2f} {m.confirmed_share:6.3f} "
        f"{m.unmapped_clustered_km2:7.2f} {m.mapped_extent:6.3f} {m.unmapped_outside_extent_km2:7.2f}  "
        f"{r.verdict:14} {flags}"
    )


print(
    f"{'':8} {'product':18} {'made from':22} {'S-1':4} {'gap h':>6} {'cov':>5} {'km²':>7} {'conf':>6} "
    f"{'clu km²':>7} {'extent':>6} {'out km²':>7}  {'verdict':14} flags / reason"
)
for x in rows:
    print(line(x))

shares = [x["report"].measurements.confirmed_share for x in assessable]
print(
    f"\nconfirmed share of the assessable maps: median {median(shares):.3f}, half the median {median(shares) / 2:.3f}"
)
for verdict in (quality.COHERENT, quality.TO_CHECK, quality.NOT_ASSESSABLE):
    print(f"{verdict:15} {sum(r.verdict == verdict for r in reports)}")

## 2. The sweep

Each setting is applied by patching the four module constants (`unittest.mock.patch.multiple`) and calling the public `quality.assess` again on the same measurements: the rules are the module's own, nothing is copied here.

In [ ]:
NAMES = ("PARTIAL_EXTENT", "PARTIAL_RATIO", "UNMAPPED_RATIO", "LITTLE_CONFIRMED")
GRID = {
    "PARTIAL_EXTENT": (0.02, 0.05, 0.1, 0.15, 0.2, 0.3),
    "PARTIAL_RATIO": (0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0),
    "UNMAPPED_RATIO": (0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0),
    "LITTLE_CONFIRMED": (0.0, 0.005, 0.01, 0.015, 0.02, 0.025, 0.05, 0.1),
}
CHOSEN = {n: getattr(quality, n) for n in NAMES}


def run(setting):
    """Number of assessable maps 'to check', whether the suspect is possibly partial, who is flagged."""
    with mock.patch.multiple(quality, **setting):
        out = [
            quality.assess(
                x["pair"].meta,
                x["report"].measurements,
                resolution_m=x["report"].resolution_m,
                has_reference=True,
            )
            for x in assessable
        ]
    flagged = [r for r in out if r.verdict == quality.TO_CHECK]
    suspect = next(r for r in out if (r.activation, r.product) == SUSPECT)
    partial = any(f.code == "possibly_partial" for f in suspect.flags)
    return len(flagged), partial, flagged


for name in NAMES:
    print(f"\n{name} (the others as in the module)")
    for value in GRID[name]:
        n, partial, flagged = run({**CHOSEN, name: value})
        mark = "within cap" if n <= CAP else "OVER CAP"
        print(
            f"  {value:6}: {n:2} to check ({mark}), suspect possibly partial: {'yes' if partial else 'no ':3}  "
            + ", ".join(f"{r.activation} {r.product}" for r in flagged)
        )

ok = total = 0
for pe in GRID["PARTIAL_EXTENT"]:
    for pr in GRID["PARTIAL_RATIO"]:
        for ur in GRID["UNMAPPED_RATIO"]:
            for lc in GRID["LITTLE_CONFIRMED"]:
                n, partial, _ = run(dict(zip(NAMES, (pe, pr, ur, lc), strict=True)))
                total += 1
                ok += n <= CAP and partial
print(f"\nfull grid: {ok} of {total} settings stay within the cap and flag the suspect")

## 3. Margins of the chosen values

How far each map sits from each threshold: the closest map on either side of a threshold says how much it could move before a verdict changes.

In [ ]:
def ratio(m, km2):
    return km2 / m.mapped_km2


def nearest(values, threshold):
    below = max((v for v in values if v < threshold), default=None)
    above = min((v for v in values if v >= threshold), default=None)
    return below, above


ms = {
    f"{x['report'].activation} {x['report'].product}": x["report"].measurements for x in assessable
}
checks = {
    "PARTIAL_EXTENT": [m.mapped_extent for m in ms.values()],
    "PARTIAL_RATIO": [ratio(m, m.unmapped_outside_extent_km2) for m in ms.values()],
    "UNMAPPED_RATIO": [ratio(m, m.unmapped_clustered_km2) for m in ms.values()],
    "LITTLE_CONFIRMED": [m.confirmed_share for m in ms.values()],
}
for name, values in checks.items():
    below, above = nearest(values, CHOSEN[name])
    print(
        f"{name:17} {CHOSEN[name]:5}: nearest map below {below:.3f}, nearest at or above {above:.3f}"
    )
s = ms[" ".join(SUSPECT)]
print(
    f"\nsuspect: extent {s.mapped_extent:.3f}, outside ratio {ratio(s, s.unmapped_outside_extent_km2):.2f}, "
    f"clustered ratio {ratio(s, s.unmapped_clustered_km2):.2f}, confirmed {s.confirmed_share:.3f}"
)

## 4. Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

COLORS = {
    quality.COHERENT: "#2a78d6",
    quality.TO_CHECK: "#eb6834",
    quality.NOT_ASSESSABLE: "#b5b5b5",
}
order = sorted(
    rows,
    key=lambda x: (
        x["report"].verdict == quality.NOT_ASSESSABLE,
        x["report"].measurements.confirmed_share,
    ),
)
fig, ax = plt.subplots(figsize=(8, 6.4))
for i, x in enumerate(order):
    r = x["report"]
    v = r.measurements.confirmed_share
    ax.barh(i, v, color=COLORS[r.verdict], height=0.7)
    note = ", ".join(f.code.replace("_", " ") for f in r.flags)
    ax.text(v + 0.008, i, f"{v:.2f}  {note}", va="center", fontsize=7.5, color="#444")
ax.set_yticks(
    range(len(order)),
    [f"{x['report'].activation} {x['report'].product}" for x in order],
    fontsize=7.5,
)
ax.invert_yaxis()
ax.axvline(quality.LITTLE_CONFIRMED, color="#444", lw=0.8, ls="--")
ax.set_xlim(0, 1.05)
ax.set_xlabel("share of the mapped water the radar confirms")
handles = [Patch(color=color, label=verdict) for verdict, color in COLORS.items()]
handles.append(Line2D([], [], color="#444", lw=0.8, ls="--", label="LITTLE_CONFIRMED"))
ax.legend(handles=handles, frameon=False, loc="upper right")
ax.grid(axis="x", color="#e5e5e5")
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
fig.text(
    0.5,
    0.005,
    "Contains modified Copernicus Sentinel data and Copernicus EMS information, © European Union",
    ha="center",
    fontsize=7,
)
fig.tight_layout(rect=(0, 0.03, 1, 1))
FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE, dpi=110)
print("saved", FIGURE)

## Reading the verdicts

* **3 of the 16 assessable maps are "to check"**, under the cap of 4; 13 are coherent. The 8 maps drawn on Sentinel-1 are not assessable: the radar would be checking the image the map was drawn on.
* **The suspect is flagged possibly partial.** EMSR664 AOI01 Pléiades (7.7 km²) spans 1.3 % of the area, and the radar sees 4.5 times as much darkened water in large patches outside it; it is also flagged for much unmapped water (the same 4.5 times). The flag holds for any `PARTIAL_EXTENT` from 0.02 to 0.3 and any `PARTIAL_RATIO` up to 4. No other assessable map has large unmapped patches outside its extent, except the COSMO-SkyMed map of the same radar image (0.3 times its mapped water).
* **Little confirmed was the weak flag, and was tightened from 0.1 to 0.01.** The radar confirms a small share of most maps drawn on other sensors and dates (median 0.10): at 0.1 half the assessable maps fell under it, at half the median (0.05) five. 0.02 would reach exactly the cap, but it falls between 0.0196 and 0.0209, so a tiny change would move a map across. At 0.01 the two maps under it sit at 0.000 and 0.007, the next one at 0.020, twice the threshold.
* **Two maps where the radar confirms almost nothing**: EMSR762 AOI07 (GeoEye-1, 0.8 km², none confirmed) and EMSR705 AOI06 (Pléiades, 3.2 km², 0.7 %, radar 19 h later). Water that receded or arrived between the two acquisitions, or lay under vegetation or in towns, fits that as well as a map to look at again. The verdict says where to look, not what is there.
* **Limits.** One known suspect, 16 assessable maps, one country, 40 m, this detector: a screening, not a validated classifier. "Coherent" means the radar does not contradict the map; it does not say the map is complete.

*Data: Copernicus EMS Rapid Mapping, © European Union; Copernicus Sentinel-1, processed by Sentinel Hub on the Copernicus Data Space Ecosystem.*